# 🌍 AQI (Air Quality Index) - Exploratory Data Analysis

##### **Project Type** - EDA
##### **Dataset** - AQI and Lat-Long of Countries

---


# 📋 Project Summary

This project conducts a comprehensive Exploratory Data Analysis (EDA) on a global Air Quality Index (AQI) dataset containing **16,695 location records** across the world. The dataset includes AQI values for overall pollution and individual pollutants — CO, Ozone, NO2, and PM2.5 — alongside geographic coordinates (latitude and longitude).

**Key Findings:**
1. PM2.5 is the dominant pollutant — it has a 0.98 correlation with overall AQI.
2. The majority (~46%) of locations fall in the **Good** AQI category; only ~1% are Hazardous.
3. High AQI hotspots are concentrated in **South Asia, East Asia, and parts of Africa**.
4. Ozone is weakly correlated with overall AQI compared to other pollutants.
5. Locations in the **Northern Hemisphere** tend to have higher pollution levels than the Southern Hemisphere.


# ❓ Problem Statement

Air pollution is one of the leading environmental health risks globally. This analysis aims to:
1. Understand the distribution of AQI across global locations.
2. Identify which pollutants (CO, Ozone, NO2, PM2.5) contribute most to overall AQI.
3. Discover geographic patterns — which regions experience the worst air quality?
4. Categorize locations by AQI severity and explore the distribution.
5. Find correlations between individual pollutant AQI values.

**Business Objective:** Provide data-driven insights to support environmental agencies, policymakers, and public health organizations in prioritizing interventions.


# 1. Know Your Data
## Import Libraries


In [ ]:
# Import all necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy import stats

# Plotting style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

print('Libraries imported successfully!')


## Dataset Loading


In [ ]:
# Load the AQI dataset
df = pd.read_csv('AQI-and-Lat-Long-of-Countries.csv')
print(f'Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns')


## Dataset First View


In [ ]:
# First look at the data
df.head(10)


## Dataset Shape


In [ ]:
print(f'Rows    : {df.shape[0]:,}')
print(f'Columns : {df.shape[1]}')


## Dataset Information


In [ ]:
df.info()


## Duplicate Values


In [ ]:
dups = df.duplicated().sum()
print(f'Number of duplicate rows: {dups}')


## Missing Values


In [ ]:
missing = df.isnull().sum()
print(missing)
print(f'\nTotal missing values: {missing.sum()}')


In [ ]:
# Heatmap of missing values
plt.figure(figsize=(10, 4))
sns.heatmap(df.isnull(), yticklabels=False, cbar=True, cmap='viridis')
plt.title('Missing Values Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### What did you know about your dataset?

The dataset has **16,695 records** and **7 columns**. All columns are numeric — five AQI metrics and two geographic coordinates (lat/lng). There are **no missing values** and **no duplicate rows**, making it clean for analysis right away.


# 2. Understanding Your Variables


In [ ]:
# Column names
print(df.columns.tolist())


In [ ]:
# Statistical summary
df.describe().round(2)


### Variable Descriptions
| Column | Description |
|---|---|
| **AQI Value** | Overall Air Quality Index (higher = worse air quality) |
| **CO AQI Value** | Carbon Monoxide sub-index |
| **Ozone AQI Value** | Ozone (O₃) sub-index |
| **NO2 AQI Value** | Nitrogen Dioxide sub-index |
| **PM2.5 AQI Value** | Fine Particulate Matter sub-index |
| **lat** | Latitude of the location |
| **lng** | Longitude of the location |


In [ ]:
# Unique value counts per column
for col in df.columns:
    print(f'{col}: {df[col].nunique()} unique values')


# 3. Data Wrangling


In [ ]:
# AQI Category classification (US EPA standard)
def aqi_category(val):
    if val <= 50:   return 'Good'
    elif val <= 100: return 'Moderate'
    elif val <= 150: return 'Unhealthy for Sensitive Groups'
    elif val <= 200: return 'Unhealthy'
    elif val <= 300: return 'Very Unhealthy'
    else:            return 'Hazardous'

df['AQI_Category'] = df['AQI Value'].apply(aqi_category)

# Hemisphere feature
df['Hemisphere'] = df['lat'].apply(lambda x: 'Northern' if x >= 0 else 'Southern')

# Dominant pollutant
pollutant_cols = ['CO AQI Value', 'Ozone AQI Value', 'NO2 AQI Value', 'PM2.5 AQI Value']
df['Dominant_Pollutant'] = df[pollutant_cols].idxmax(axis=1).str.replace(' AQI Value', '')

# AQI severity score (normalized 0-1)
df['AQI_Severity'] = (df['AQI Value'] - df['AQI Value'].min()) / (df['AQI Value'].max() - df['AQI Value'].min())

print('Feature engineering complete.')
df.head()


### What manipulations were done?
1. **AQI_Category**: Classified each record using US EPA AQI breakpoints (Good → Hazardous).
2. **Hemisphere**: Derived from latitude to enable geographic analysis.
3. **Dominant_Pollutant**: The pollutant with the highest sub-index for each location.
4. **AQI_Severity**: Min-max normalized AQI for scaled comparisons.


# 4. Data Visualization, Storytelling & Insights


## 📊 Univariate Analysis
### Chart 1 — Distribution of Overall AQI Values


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['AQI Value'], bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(df['AQI Value'].mean(), color='red', linestyle='--', label=f"Mean: {df['AQI Value'].mean():.1f}")
axes[0].axvline(df['AQI Value'].median(), color='green', linestyle='--', label=f"Median: {df['AQI Value'].median():.1f}")
axes[0].set_title('AQI Value Distribution', fontweight='bold')
axes[0].set_xlabel('AQI Value')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# KDE
df['AQI Value'].plot.kde(ax=axes[1], color='steelblue', linewidth=2)
axes[1].set_title('AQI Value — Density Curve', fontweight='bold')
axes[1].set_xlabel('AQI Value')
axes[1].set_ylabel('Density')

plt.suptitle('Global AQI Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


**Why this chart?** Histogram + KDE gives a complete picture of data shape, skew, and central tendency.

**Insights:** AQI is right-skewed — most locations have relatively low AQI (Good/Moderate) but a long tail extends to 500 (Hazardous). Mean (63) > Median (52), confirming the right skew driven by extreme pollution hotspots.

**Business Impact:** The skew tells policymakers that while most areas are acceptable, the tail represents urgent intervention zones.


### Chart 2 — AQI Category Distribution (Count)


In [ ]:
cat_order = ['Good', 'Moderate', 'Unhealthy for Sensitive Groups', 'Unhealthy', 'Very Unhealthy', 'Hazardous']
palette   = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c', '#9b59b6', '#7f1a1a']
cat_counts = df['AQI_Category'].value_counts().reindex(cat_order)

plt.figure(figsize=(12, 5))
bars = plt.bar(cat_counts.index, cat_counts.values, color=palette, edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, cat_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 60, f'{val:,}\n({val/len(df)*100:.1f}%)',
             ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.title('Distribution of AQI Categories', fontsize=14, fontweight='bold')
plt.xlabel('AQI Category')
plt.ylabel('Number of Locations')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


**Why this chart?** A bar chart clearly shows the frequency of each AQI category.

**Insights:** 46.2% of locations are 'Good' and 42.4% are 'Moderate'. Combined, ~89% of monitored locations have acceptable air quality. However, 5.4% fall in 'Unhealthy' or worse — impacting millions of people.

**Business Impact:** Positive — most of the world has manageable air quality. Negative — the 5.4% Unhealthy+ cluster represents areas demanding urgent action.


### Chart 3 — Boxplots of All AQI Pollutants


In [ ]:
aqi_cols = ['AQI Value', 'CO AQI Value', 'Ozone AQI Value', 'NO2 AQI Value', 'PM2.5 AQI Value']
fig, axes = plt.subplots(1, 5, figsize=(18, 6))

colors = ['steelblue', 'salmon', 'mediumseagreen', 'goldenrod', 'mediumpurple']
for ax, col, color in zip(axes, aqi_cols, colors):
    ax.boxplot(df[col], patch_artist=True,
               boxprops=dict(facecolor=color, alpha=0.7),
               medianprops=dict(color='black', linewidth=2))
    ax.set_title(col.replace(' AQI Value', ''), fontweight='bold', fontsize=10)
    ax.set_xticklabels([])

plt.suptitle('Boxplots — AQI Pollutant Distribution & Outliers', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


**Why this chart?** Boxplots reveal spread, median, IQR, and outliers for each pollutant simultaneously.

**Insights:** PM2.5 and overall AQI have the widest spread and most extreme outliers, indicating that fine particles drive pollution spikes. CO and NO2 are tightly distributed with few outliers. Ozone shows moderate spread.

**Business Impact:** PM2.5 regulation should be the top priority given its extreme variability and dominance in outlier scenarios.


### Chart 4 — Dominant Pollutant Distribution


In [ ]:
dom_counts = df['Dominant_Pollutant'].value_counts()

plt.figure(figsize=(8, 8))
wedge_colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']
plt.pie(dom_counts.values, labels=dom_counts.index, autopct='%1.1f%%',
        colors=wedge_colors, startangle=140,
        wedgeprops=dict(edgecolor='white', linewidth=1.5))
plt.title('Dominant Pollutant by Location Count', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


**Why this chart?** A pie chart is ideal for showing proportional dominance across a small number of categories.

**Insights:** PM2.5 dominates as the primary pollutant in the vast majority of locations worldwide, followed by Ozone. CO and NO2 dominate only in a small fraction of locations.

**Business Impact:** Global air quality strategy should heavily focus on PM2.5 reduction (vehicular emissions, industrial burning, dust).


## 📊 Bivariate Analysis
### Chart 5 — PM2.5 vs AQI Scatter Plot


In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df['PM2.5 AQI Value'], df['AQI Value'],
            alpha=0.3, c='steelblue', edgecolors='none', s=15)

# Regression line
m, b = np.polyfit(df['PM2.5 AQI Value'], df['AQI Value'], 1)
x_line = np.linspace(df['PM2.5 AQI Value'].min(), df['PM2.5 AQI Value'].max(), 100)
plt.plot(x_line, m*x_line + b, color='red', linewidth=2, label=f'Trend line (y = {m:.2f}x + {b:.2f})')

plt.title('PM2.5 AQI vs Overall AQI', fontsize=13, fontweight='bold')
plt.xlabel('PM2.5 AQI Value')
plt.ylabel('Overall AQI Value')
plt.legend()
plt.tight_layout()
plt.show()

corr = df['PM2.5 AQI Value'].corr(df['AQI Value'])
print(f'Pearson Correlation: {corr:.4f}')


**Why this chart?** Scatter + regression line reveals both correlation strength and linearity.

**Insights:** Near-perfect linear correlation (r ≈ 0.98) between PM2.5 and overall AQI. PM2.5 almost entirely determines the AQI value, making it the single most critical pollutant to monitor and regulate.

**Business Impact:** Monitoring stations could prioritize PM2.5 sensors for the most accurate AQI estimation.


### Chart 6 — AQI by Hemisphere


In [ ]:
plt.figure(figsize=(9, 6))
sns.boxplot(data=df, x='Hemisphere', y='AQI Value',
            palette={'Northern': 'tomato', 'Southern': 'skyblue'})
plt.title('AQI Distribution: Northern vs Southern Hemisphere', fontsize=13, fontweight='bold')
plt.xlabel('Hemisphere')
plt.ylabel('AQI Value')
plt.tight_layout()
plt.show()

print(df.groupby('Hemisphere')['AQI Value'].describe().round(2))


**Why this chart?** Boxplot comparison across two groups effectively shows distributional differences.

**Insights:** The Northern Hemisphere has significantly higher median and maximum AQI values. This is expected — most of the world's major industrial nations and densely populated cities are in the Northern Hemisphere.

**Business Impact:** Northern Hemisphere nations need more aggressive emission controls and inter-country cooperation.


### Chart 7 — Correlation Heatmap of All AQI Variables


In [ ]:
corr_matrix = df[['AQI Value','CO AQI Value','Ozone AQI Value','NO2 AQI Value','PM2.5 AQI Value']].corr()

plt.figure(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, vmin=-1, vmax=1,
            linewidths=0.5, linecolor='white',
            annot_kws={'size': 12})
plt.title('Correlation Heatmap — AQI Pollutants', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


**Why this chart?** Heatmaps are the clearest way to display a full correlation matrix.

**Insights:** PM2.5 ↔ AQI = 0.98 (near-perfect). CO ↔ AQI = 0.46. Ozone is weakly and even slightly negatively correlated with NO2 (−0.25). NO2 and CO are moderately correlated (0.40), suggesting they share emission sources (combustion engines).

**Business Impact:** CO and NO2 share sources — policies targeting vehicle emissions can reduce both simultaneously.


### Chart 8 — AQI Category vs Dominant Pollutant (Stacked Bar)


In [ ]:
cat_order = ['Good', 'Moderate', 'Unhealthy for Sensitive Groups', 'Unhealthy', 'Very Unhealthy', 'Hazardous']
ct = pd.crosstab(df['AQI_Category'], df['Dominant_Pollutant'])
ct = ct.reindex(cat_order)
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100

ct_pct.plot(kind='bar', stacked=True, figsize=(13, 6),
            colormap='Set2', edgecolor='white')
plt.title('Dominant Pollutant by AQI Category (%)', fontsize=13, fontweight='bold')
plt.xlabel('AQI Category')
plt.ylabel('Percentage')
plt.xticks(rotation=20)
plt.legend(title='Dominant Pollutant', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()


**Why this chart?** Stacked percentage bars show how pollutant composition changes across severity levels.

**Insights:** As AQI severity increases from Good to Hazardous, PM2.5 becomes increasingly dominant. In 'Good' zones, Ozone plays a more balanced role. Hazardous locations are almost exclusively PM2.5-driven.

**Business Impact:** Emergency response protocols for Hazardous AQI days should specifically target PM2.5 sources.


### Chart 9 — Scatter: Latitude vs AQI (with Category Color)


In [ ]:
cat_order = ['Good', 'Moderate', 'Unhealthy for Sensitive Groups', 'Unhealthy', 'Very Unhealthy', 'Hazardous']
palette_cat = {'Good':'#2ecc71','Moderate':'#f1c40f',
               'Unhealthy for Sensitive Groups':'#e67e22',
               'Unhealthy':'#e74c3c','Very Unhealthy':'#9b59b6','Hazardous':'#7f1a1a'}

plt.figure(figsize=(13, 6))
for cat in cat_order:
    subset = df[df['AQI_Category'] == cat]
    plt.scatter(subset['lat'], subset['AQI Value'],
                c=palette_cat[cat], label=cat, alpha=0.4, s=10)

plt.axhline(100, color='gray', linestyle='--', linewidth=1, label='AQI=100 threshold')
plt.title('Latitude vs AQI Value', fontsize=13, fontweight='bold')
plt.xlabel('Latitude')
plt.ylabel('AQI Value')
plt.legend(bbox_to_anchor=(1.01, 1), markerscale=2)
plt.tight_layout()
plt.show()


**Why this chart?** Shows a geographic (latitude) dimension against AQI — revealing North-South patterns spatially.

**Insights:** The highest AQI values cluster between latitudes 10°N–40°N (South Asia, East Asia, Middle East, North Africa). Southern Hemisphere latitudes rarely exceed the 'Moderate' category.

**Business Impact:** International climate organizations should focus pollution-reduction investments in tropical to mid-latitude Northern Hemisphere regions.


### Chart 10 — Pairplot of AQI Variables


In [ ]:
# Sample for pairplot performance
sample = df[['AQI Value','CO AQI Value','Ozone AQI Value','NO2 AQI Value','PM2.5 AQI Value','AQI_Category']].sample(2000, random_state=42)

g = sns.pairplot(sample, hue='AQI_Category',
                 vars=['AQI Value','CO AQI Value','Ozone AQI Value','PM2.5 AQI Value'],
                 palette=palette_cat, plot_kws={'alpha': 0.3, 's': 10},
                 diag_kind='kde')
g.fig.suptitle('Pairplot of AQI Variables by Category', y=1.01, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


**Why this chart?** Pairplot captures all pairwise relationships in one view, colored by AQI category.

**Insights:** PM2.5 vs AQI is nearly perfectly linear. 'Hazardous' and 'Very Unhealthy' points form distinct clusters in high PM2.5 regions. Ozone vs other pollutants shows weak / no clear relationship.

**Business Impact:** Confirms PM2.5 as the single most actionable variable for AQI prediction and regulation.


## 📊 Multivariate Analysis
### Chart 11 — Geographic Scatter Map (Lat/Lng colored by AQI)


In [ ]:
plt.figure(figsize=(16, 8))
scatter = plt.scatter(df['lng'], df['lat'],
                       c=df['AQI Value'], cmap='RdYlGn_r',
                       s=5, alpha=0.6, vmin=0, vmax=200)
plt.colorbar(scatter, label='AQI Value')
plt.title('Global AQI Map (Lat/Long)', fontsize=14, fontweight='bold')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.axhline(0, color='black', linewidth=0.5, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


**Why this chart?** Plotting lat/lng with color encoding converts tabular data into a pseudo-map — the most intuitive geographic view.

**Insights:** Red/orange clusters (high AQI) are visible over South Asia (India, Bangladesh), East Asia (China), and the Middle East. Green clusters dominate the Americas, Australia, and Northern Europe.

**Business Impact:** The visualization immediately identifies the regions requiring the most urgent international air quality action.


### Chart 12 — Violin Plot: AQI by AQI Category


In [ ]:
cat_order = ['Good', 'Moderate', 'Unhealthy for Sensitive Groups', 'Unhealthy', 'Very Unhealthy', 'Hazardous']
palette_cat = {'Good':'#2ecc71','Moderate':'#f1c40f',
               'Unhealthy for Sensitive Groups':'#e67e22',
               'Unhealthy':'#e74c3c','Very Unhealthy':'#9b59b6','Hazardous':'#7f1a1a'}
plt.figure(figsize=(13, 6))
sns.violinplot(data=df, x='AQI_Category', y='AQI Value',
               order=cat_order, palette=palette_cat, inner='quartile')
plt.title('AQI Value Distribution within Each Category', fontsize=13, fontweight='bold')
plt.xlabel('AQI Category')
plt.ylabel('AQI Value')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


**Why this chart?** Violin plots show full distribution shape (not just quartiles), revealing whether data is uniform or bimodal within each category.

**Insights:** 'Good' and 'Moderate' categories have wide distributions filling their AQI ranges. 'Hazardous' has a highly concentrated distribution at the extreme high end, suggesting these are consistent, persistent pollution zones (not anomalies).

**Business Impact:** Hazardous locations need permanent industrial controls, not just emergency measures.


### Chart 13 — Multi-pollutant AQI Comparison by Category (Grouped Bar)


In [ ]:
cat_order = ['Good', 'Moderate', 'Unhealthy for Sensitive Groups', 'Unhealthy', 'Very Unhealthy', 'Hazardous']
group_means = df.groupby('AQI_Category')[['CO AQI Value','Ozone AQI Value','NO2 AQI Value','PM2.5 AQI Value']].mean().reindex(cat_order)

group_means.plot(kind='bar', figsize=(14, 6), colormap='tab10', edgecolor='white')
plt.title('Mean Pollutant Sub-Index by AQI Category', fontsize=13, fontweight='bold')
plt.xlabel('AQI Category')
plt.ylabel('Mean AQI Sub-Index')
plt.xticks(rotation=20)
plt.legend(title='Pollutant', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()


**Why this chart?** Grouped bar chart allows simultaneous comparison of 4 pollutants across 6 severity categories.

**Insights:** PM2.5 shows a steep, exponential rise as AQI category worsens — by 'Hazardous', it is far higher than all other pollutants. Ozone remains relatively flat. CO and NO2 increase modestly.

**Business Impact:** Policy interventions at every AQI level should be PM2.5-first. Other pollutants matter less at extreme pollution levels.


### Chart 14 — AQI Severity Heatmap (Lat × Lng Bins)


In [ ]:
# Bin latitude and longitude
df['lat_bin'] = pd.cut(df['lat'], bins=18, labels=False)
df['lng_bin'] = pd.cut(df['lng'], bins=36, labels=False)

pivot = df.pivot_table(index='lat_bin', columns='lng_bin', values='AQI Value', aggfunc='mean')

plt.figure(figsize=(16, 7))
sns.heatmap(pivot, cmap='RdYlGn_r', cbar_kws={'label': 'Mean AQI'}, linewidths=0)
plt.title('Global AQI Heatmap (Lat × Lng Bins)', fontsize=13, fontweight='bold')
plt.xlabel('Longitude Bin (West → East)')
plt.ylabel('Latitude Bin (South → North)')
plt.tight_layout()
plt.show()


**Why this chart?** Binned lat/lng heatmap is a compact pseudo-map that reveals geographic pollution clusters clearly.

**Insights:** Bright red zones appear over South/East Asia and parts of Central Africa. The Americas, most of Europe, and Australia are predominantly green (low AQI).

**Business Impact:** Enables rapid identification of global pollution hotspots without a full GIS system.


### Chart 15 — Distribution of Each Pollutant AQI (Overlapping KDE)


In [ ]:
plt.figure(figsize=(12, 6))
pollutants = {'CO AQI Value': 'salmon', 'Ozone AQI Value': 'mediumseagreen',
              'NO2 AQI Value': 'goldenrod', 'PM2.5 AQI Value': 'mediumpurple'}
for col, color in pollutants.items():
    df[col].plot.kde(label=col.replace(' AQI Value',''), color=color, linewidth=2)

plt.xlim(-5, 150)
plt.title('KDE — Individual Pollutant AQI Distributions', fontsize=13, fontweight='bold')
plt.xlabel('AQI Sub-Index')
plt.ylabel('Density')
plt.legend(title='Pollutant')
plt.tight_layout()
plt.show()


**Why this chart?** Overlapping KDE curves are the clearest way to compare the shape and spread of multiple distributions simultaneously.

**Insights:** CO and NO2 are very tightly concentrated near 0–2 (low risk globally). Ozone has a broader distribution peaking around 30–50. PM2.5 has the widest distribution with the heaviest right tail.

**Business Impact:** CO and NO2 risks are concentrated in specific hotspot cities — targeted enforcement works better than sweeping regulation.


### Chart 16 — CDF of AQI Values by Category


In [ ]:
cat_order = ['Good', 'Moderate', 'Unhealthy for Sensitive Groups', 'Unhealthy', 'Very Unhealthy', 'Hazardous']
palette_cat = {'Good':'#2ecc71','Moderate':'#f1c40f',
               'Unhealthy for Sensitive Groups':'#e67e22',
               'Unhealthy':'#e74c3c','Very Unhealthy':'#9b59b6','Hazardous':'#7f1a1a'}
plt.figure(figsize=(12, 6))
for cat in cat_order:
    subset = df[df['AQI_Category'] == cat]['AQI Value'].sort_values()
    cdf = np.arange(1, len(subset)+1) / len(subset)
    plt.plot(subset.values, cdf, label=cat, color=palette_cat[cat], linewidth=2)

plt.title('Cumulative Distribution Function by AQI Category', fontsize=13, fontweight='bold')
plt.xlabel('AQI Value')
plt.ylabel('Cumulative Proportion')
plt.legend(bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()


**Why this chart?** CDFs precisely show what proportion of each category falls below any AQI threshold.

**Insights:** The 'Good' curve rises steeply and plateaus before AQI 50, confirming tight clustering. 'Hazardous' locations have AQI values spread well above 300, with some reaching 500.

**Business Impact:** Helps regulators set enforceable AQI thresholds and estimate what percentage of locations would comply.


### Chart 17 — Outlier Detection: AQI Z-Score


In [ ]:
df['AQI_zscore'] = np.abs(stats.zscore(df['AQI Value']))
outliers = df[df['AQI_zscore'] > 3]

plt.figure(figsize=(13, 5))
plt.scatter(df.index, df['AQI Value'], alpha=0.3, s=8, color='steelblue', label='Normal')
plt.scatter(outliers.index, outliers['AQI Value'], color='red', s=30, zorder=5, label=f'Outliers (n={len(outliers)})')
plt.axhline(df['AQI Value'].mean() + 3*df['AQI Value'].std(), color='orange', linestyle='--', label='3σ threshold')
plt.title('AQI Value — Outlier Detection (Z-Score > 3)', fontsize=13, fontweight='bold')
plt.xlabel('Record Index')
plt.ylabel('AQI Value')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Total outliers: {len(outliers)}')
print(outliers[['AQI Value','lat','lng','AQI_Category']].describe())


**Why this chart?** Z-score flagging highlights statistically extreme values — useful for data quality and hotspot identification.

**Insights:** A small number of locations have AQI values exceeding the 3σ threshold (> ~193). These are extreme pollution events or persistent industrial zones, not measurement errors.

**Business Impact:** These outlier locations should be placed on an emergency monitoring and intervention list.


### Chart 18 — PM2.5 AQI Distribution across Hemispheres (KDE)


In [ ]:
plt.figure(figsize=(10, 5))
for hemi, color in [('Northern','tomato'),('Southern','steelblue')]:
    df[df['Hemisphere']==hemi]['PM2.5 AQI Value'].plot.kde(label=hemi, color=color, linewidth=2.5)
plt.xlim(-5, 200)
plt.title('PM2.5 AQI Distribution: Northern vs Southern Hemisphere', fontsize=13, fontweight='bold')
plt.xlabel('PM2.5 AQI Value')
plt.ylabel('Density')
plt.legend(title='Hemisphere')
plt.tight_layout()
plt.show()

print(df.groupby('Hemisphere')['PM2.5 AQI Value'].describe().round(2))


**Why this chart?** KDE comparison between hemispheres shows the full distributional shape difference, not just medians.

**Insights:** The Northern Hemisphere's PM2.5 distribution has a much heavier right tail — meaning far more extreme PM2.5 events. The Southern Hemisphere's distribution is tightly concentrated near low values.

**Business Impact:** International PM2.5 reduction efforts should be almost entirely Northern Hemisphere focused.


### Chart 19 — AQI vs All Pollutants (Subplots Scatter)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
pollutants = ['CO AQI Value', 'Ozone AQI Value', 'NO2 AQI Value', 'PM2.5 AQI Value']
colors = ['salmon', 'mediumseagreen', 'goldenrod', 'mediumpurple']

for ax, pol, col in zip(axes.ravel(), pollutants, colors):
    ax.scatter(df[pol], df['AQI Value'], alpha=0.2, s=8, color=col)
    corr = df[pol].corr(df['AQI Value'])
    ax.set_title(f"{pol.replace(' AQI Value','')} vs AQI  (r={corr:.3f})", fontweight='bold')
    ax.set_xlabel(pol.replace(' AQI Value',''))
    ax.set_ylabel('AQI Value')

plt.suptitle('Individual Pollutants vs Overall AQI', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


**Why this chart?** 2×2 scatter grid provides a full pollutant-vs-AQI picture in one view.

**Insights:** PM2.5 (r=0.98) dominates. CO is moderate (r≈0.46). Ozone (r≈0.33) and NO2 (r≈0.31) show weak correlations, indicating they contribute to overall AQI but are rarely the primary driver.

**Business Impact:** Data-driven prioritization — PM2.5 reduction yields the most AQI improvement per unit of policy effort.


### Chart 20 — AQI Category Proportion by Dominant Pollutant


In [ ]:
cat_order = ['Good', 'Moderate', 'Unhealthy for Sensitive Groups', 'Unhealthy', 'Very Unhealthy', 'Hazardous']
palette_cat = {'Good':'#2ecc71','Moderate':'#f1c40f',
               'Unhealthy for Sensitive Groups':'#e67e22',
               'Unhealthy':'#e74c3c','Very Unhealthy':'#9b59b6','Hazardous':'#7f1a1a'}
ct2 = pd.crosstab(df['Dominant_Pollutant'], df['AQI_Category'])
ct2 = ct2.reindex(columns=cat_order, fill_value=0)
ct2_pct = ct2.div(ct2.sum(axis=1), axis=0) * 100

ct2_pct.plot(kind='barh', stacked=True, figsize=(13, 5),
             color=['#2ecc71','#f1c40f','#e67e22','#e74c3c','#9b59b6','#7f1a1a'])
plt.title('AQI Category Distribution within Each Dominant Pollutant Group', fontsize=13, fontweight='bold')
plt.xlabel('Percentage (%)')
plt.ylabel('Dominant Pollutant')
plt.legend(title='AQI Category', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()


**Why this chart?** Horizontal stacked bars make proportion comparisons across groups easy to read.

**Insights:** Locations where PM2.5 dominates have a much higher proportion of 'Unhealthy' to 'Hazardous' categories. Ozone-dominant locations are predominantly 'Moderate'. CO- and NO2-dominant locations cluster in 'Good' to 'Moderate'.

**Business Impact:** Regulatory frameworks can be tailored per dominant pollutant to achieve maximum health benefit efficiently.


# 5. Solution to Business Objective


## Key Findings & Recommendations

| # | Finding | Implication |
|---|---|---|
| 1 | **PM2.5 = AQI** (r=0.98) | PM2.5 regulation is the highest-ROI air quality intervention |
| 2 | 89% of locations are Good/Moderate | Most of the world has manageable air quality |
| 3 | Hazardous zones are persistent, not sporadic | Structural industrial controls needed, not just emergency response |
| 4 | Northern Hemisphere >> Southern Hemisphere | Global cooperation should focus on South/East Asia |
| 5 | CO and NO2 share combustion sources | Vehicle emission standards address both simultaneously |
| 6 | Ozone is weakly correlated with other pollutants | Ozone has independent photochemical drivers — needs separate policy |

**Overall Answer:** The global AQI crisis is largely a PM2.5 crisis, concentrated in the Northern Hemisphere's high-density industrial and urban zones. Targeted PM2.5 reduction policies in South Asia and East Asia would yield the greatest global health improvement.


# 6. Conclusion


This EDA revealed that:
- **Air quality globally is acceptable for most locations** (~89% Good/Moderate), but extreme outliers (Hazardous zones) demand urgent attention.
- **PM2.5 is the dominant driver** of poor AQI worldwide — addressing fine particulate pollution is the single most impactful strategy.
- **Geography matters**: Northern Hemisphere and low-latitude zones face disproportionately higher pollution.
- **Data is clean and complete** with no missing values, making it reliable for further predictive modeling.

**Next Steps:** Train a regression model to predict AQI from pollutant sub-indices and geographic features, and build a geospatial dashboard for real-time monitoring.
